In [ ]:
%pip install pandas

In [4]:
PROJECT_DIR = "/Users/mattan/thesis/"

In [1]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.en", cache_dir="~/thesis/RAG/data")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import glob
import requests
from pprint import pprint
import json
import shutil
import os

In [5]:
# Define the path to your CSV files
path = f"{PROJECT_DIR}/RAG/SolarSystem_pages/"

# Use glob to get a list of all CSV files in the specified path
csv_files = glob.glob(path + "datapage_id_title_batch_*.csv")

# Initialize an empty list to hold the DataFrames
dataframes = []

# Loop through the list of CSV files and read each one into a DataFrame
for file in csv_files:
    df = pd.read_csv(file, header=0)
    dataframes.append(df)

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)


In [44]:
combined_df[combined_df['id'] == 63087910]

,id,title,category,category_depth,Unnamed: 0
176022,63087910,$50SAT,Category:CubeSats,4,NaN
190014,63087910,$50SAT,Category:Amateur radio satellites,4,NaN


In [65]:
df_unique = combined_df.drop_duplicates(subset='id', keep='first')
# filtered_df = combined_df.loc[combined_df.groupby('title')['category_depth'].idxmin()].reset_index()
df_unique
# display(filtered_df.where(combined_df['title'] == 'Earth'))

,id,title,category,category_depth,Unnamed: 0
0,378782,Shivini,Category:Solar gods,5,NaN
1,58668847,Šimige,Category:Solar gods,5,NaN
2,20608974,Sol (Roman mythology),Category:Solar gods,5,NaN
3,1214754,Sol Invictus,Category:Solar gods,5,NaN
4,50360626,Sué,Category:Solar gods,5,NaN
...,...,...,...,...,...
226340,22136062,March 1914 lunar eclipse,Category:20th-century lunar eclipses,5,NaN
226345,55524155,April 1930 lunar eclipse,Category:20th-century lunar eclipses,5,NaN
226349,22136057,March 1932 lunar eclipse,Category:20th-century lunar eclipses,5,NaN
226352,55524192,March 1933 lunar eclipse,Category:20th-century lunar eclipses,5,NaN


In [53]:
df_unique[df_unique['title'] == "Titan (moon)"]

,id,title,category,category_depth,Unnamed: 0
75420,47402,Titan (moon),Category:Moons with a prograde orbit,3,NaN


In [56]:
df_unique = df_unique.sort_values(by='category_depth', ascending=True)
df_unique

,id,title,category,category_depth,Unnamed: 0
33474,3475555,(55637) 2002 UX25,Category:Solar System,0,NaN
33638,48510,Terrestrial planet,Category:Solar System,0,NaN
33637,26751,Sun,Category:Solar System,0,NaN
33636,77178,Spaceflight,Category:Solar System,0,NaN
33635,113496,Space weather,Category:Solar System,0,NaN
...,...,...,...,...,...
13967,57893136,34215 Stutigarg,Category:Background asteroids,5,NaN
13966,57893118,34208 Danielzhang,Category:Background asteroids,5,NaN
13965,57893109,34206 Zhiyuewang,Category:Background asteroids,5,NaN
13979,57899254,34249 Leolo,Category:Background asteroids,5,NaN


In [63]:
relevant_wiki_id_list = df_unique.id.tolist()
relevant_wiki_id_set = {str(title_id) for title_id in relevant_wiki_id_list}

len(relevant_wiki_id_set)
"1559901" in relevant_wiki_id_set

True

In [ ]:
relevant_wiki_id_list

In [67]:
astronomy_wiki = ds['train'].filter(lambda x: x['id'] in relevant_wiki_id_set)
astronomy_wiki

Filter:   0%|          | 0/6407814 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 24325
})

In [68]:
from datasets import DatasetDict

astronomy_wiki_ds = DatasetDict({
    "train": astronomy_wiki
})
astronomy_wiki_ds

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 24325
    })
})

In [81]:
astronomy_wiki_df = pd.DataFrame(astronomy_wiki_ds['train'])
astronomy_wiki_df['id'] = astronomy_wiki_df['id'].astype(int)
astronomy_wiki_df = astronomy_wiki_df.sort_values(by="id", ascending=False)

In [82]:
astronomy_wiki_df

,id,url,title,text
18064,75196950,https://en.wikipedia.org/wiki/SROSS-C2,SROSS-C2,SROSS-C2 or Stretched Rohini Satellite Series ...
18063,75168205,https://en.wikipedia.org/wiki/Solaris%20%28sol...,Solaris (solar power),SOLARIS is a space-based solar power (SBSP) pr...
18062,75157611,https://en.wikipedia.org/wiki/Solar%20eclipse%...,"Solar eclipse of November 22, 1900",An annular solar eclipse occurred on November ...
18061,75157481,https://en.wikipedia.org/wiki/Gods%20in%20Wedlock,Gods in Wedlock,Gods in Wedlock is a 1942 Australian radio pla...
18060,75156842,https://en.wikipedia.org/wiki/Hmong%20calendar,Hmong calendar,The Hmong calendar (Pahawh: ; RPA: Hmoob daim ...
...,...,...,...,...
4,689,https://en.wikipedia.org/wiki/Asia,Asia,"Asia ( , ) is the largest continent in the wo..."
3,663,https://en.wikipedia.org/wiki/Apollo%208,Apollo 8,"Apollo 8 (December 21–27, 1968) was the first ..."
2,662,https://en.wikipedia.org/wiki/Apollo%2011,Apollo 11,"Apollo 11 (July 16–24, 1969) was the American ..."
1,624,https://en.wikipedia.org/wiki/Alaska,Alaska,Alaska ( ) is a non-contiguous U.S. state on t...


In [83]:
astronomy_wiki_ds.save_to_disk(f"{PROJECT_DIR}RAG/solarsystem-ds/")

Saving the dataset (0/1 shards):   0%|          | 0/24325 [00:00<?, ? examples/s]